# Murphy 1 · Chapter 13 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M13). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 13.1 · Solve XOR with a 2-2-1 network by hand  
*Section §13.2 · code*

Train a network with 2 tanh hidden units and a sigmoid output on the four XOR points, writing forward and backward passes yourself. Run it from two random starts (seeds 0 and 3) and compare the final loss and predictions.

<details><summary><b>Hint</b></summary>

Backward: dz2 = a − y; dW2 = hᵀdz2; dh = dz2 W2ᵀ; dz1 = dh ⊙ (1 − h²); dW1 = Xᵀdz1.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1.]]); y = np.array([[0], [1], [1], [0.]])
sig = lambda z: 1 / (1 + np.exp(-z))
for seed in (0, 3):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1, (2, 2)); b1 = np.zeros(2); W2 = rng.normal(0, 1, (2, 1)); b2 = np.zeros(1)
    for step in range(5001):
        h = np.tanh(X @ W1 + b1); a = sig(h @ W2 + b2)
        dz2 = (a - y) / 4
        dW2 = h.T @ dz2; db2 = dz2.sum(0)
        dz1 = ...  # TODO: blame through W2, then through tanh
        dW1 = X.T @ dz1; db1 = dz1.sum(0)
        for p, g in ((W1, dW1), (b1, db1), (W2, dW2), (b2, db2)): p -= 1.0 * g
    loss = -np.mean(y * np.log(a) + (1 - y) * np.log(1 - a))
    print(f"seed {seed}: final loss {loss:.4f}  predictions {a.ravel().round(2)}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1.]]); y = np.array([[0], [1], [1], [0.]])
sig = lambda z: 1 / (1 + np.exp(-z))
for seed in (0, 3):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1, (2, 2)); b1 = np.zeros(2); W2 = rng.normal(0, 1, (2, 1)); b2 = np.zeros(1)
    for step in range(5001):
        h = np.tanh(X @ W1 + b1); a = sig(h @ W2 + b2)
        dz2 = (a - y) / 4
        dW2 = h.T @ dz2; db2 = dz2.sum(0)
        dz1 = (dz2 @ W2.T) * (1 - h ** 2)  # blame through W2, then through tanh
        dW1 = X.T @ dz1; db1 = dz1.sum(0)
        for p, g in ((W1, dW1), (b1, db1), (W2, dW2), (b2, db2)): p -= 1.0 * g
    loss = -np.mean(y * np.log(a) + (1 - y) * np.log(1 - a))
    print(f"seed {seed}: final loss {loss:.4f}  predictions {a.ravel().round(2)}")

<details><summary><b>Answer and explanation</b></summary>

Seed 0 solves it: loss ≈ 0.0006, predictions [0, 1, 1, 0]. One hidden unit learns roughly OR, the other roughly AND, and the output computes 'OR and not AND'. Seed 3 gets stuck at loss ≈ 0.347 with predictions [0, 1, 0.5, 0.5]: it gets two points right and shrugs on the other two. More steps won't fix it; it's a local minimum. That's non-convexity (ch. 8) in its smallest form: about half of random starts fail with only 2 hidden units. Fixes: more hidden units (more ways downhill) or several random restarts (Murphy §13.2.1).

Expected output:
```
seed 0: final loss 0.0006  predictions [0. 1. 1. 0.]
seed 3: final loss 0.3468  predictions [0.  1.  0.5 0.5]
```

</details>

## 13.2 · Vanishing gradients, measured  
*Section §13.4 · code*

Push a signal through 30 layers of width 100 and measure the gradient norm reaching layer 1, for: sigmoid with standard init, tanh with Xavier init, and ReLU with He init.

<details><summary><b>Hint</b></summary>

Backward through a layer: g ← Wᵀ(g ⊙ φ′(z)). Track ‖g‖ at the first layer.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(6)
n, L = 100, 30
acts = {"sigmoid, N(0,1/n)": (lambda z: 1 / (1 + np.exp(-z)), lambda a, z: a * (1 - a), 1.0),
        "tanh, Xavier":      (np.tanh, lambda a, z: 1 - a ** 2, 1.0),
        "ReLU, He":          (lambda z: np.maximum(z, 0), lambda a, z: (z > 0) * 1.0, 2.0)}
for name, (f, df, gain) in acts.items():
    Ws = [rng.normal(0, np.sqrt(gain / n), (n, n)) for _ in range(L)]
    x = rng.normal(size=n); cache = []
    for W in Ws:
        z = W @ x; a = f(z); cache.append((z, a)); x = a
    g = np.ones(n)
    for W, (z, a) in zip(reversed(Ws), reversed(cache)):
        g = ...  # TODO: chain rule through one layer
    print(f"{name:20s} gradient norm at layer 1: {np.linalg.norm(g):.3e}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(6)
n, L = 100, 30
acts = {"sigmoid, N(0,1/n)": (lambda z: 1 / (1 + np.exp(-z)), lambda a, z: a * (1 - a), 1.0),
        "tanh, Xavier":      (np.tanh, lambda a, z: 1 - a ** 2, 1.0),
        "ReLU, He":          (lambda z: np.maximum(z, 0), lambda a, z: (z > 0) * 1.0, 2.0)}
for name, (f, df, gain) in acts.items():
    Ws = [rng.normal(0, np.sqrt(gain / n), (n, n)) for _ in range(L)]
    x = rng.normal(size=n); cache = []
    for W in Ws:
        z = W @ x; a = f(z); cache.append((z, a)); x = a
    g = np.ones(n)
    for W, (z, a) in zip(reversed(Ws), reversed(cache)):
        g = W.T @ (g * df(a, z))  # chain rule through one layer
    print(f"{name:20s} gradient norm at layer 1: {np.linalg.norm(g):.3e}")

<details><summary><b>Answer and explanation</b></summary>

The gradient starts with norm 10 at the top. With sigmoid it arrives at layer 1 as about 10⁻¹⁸: each layer multiplies by slopes of at most 0.25, and 30 such factors wipe it out. Tanh with Xavier keeps it around 1, and ReLU with He around 40, the same order of magnitude as where it started. ReLU's slope is exactly 1 for active units, and the 2/n scaling compensates for the half that are off (Murphy §13.4.2–13.4.5).

Expected output:
```
sigmoid, N(0,1/n)    gradient norm at layer 1: 1.889e-18
tanh, Xavier         gradient norm at layer 1: 1.094e+00
ReLU, He             gradient norm at layer 1: 4.324e+01
```

</details>

## 13.3 · Why dropout divides by (1 − p)  
*Section §13.5 · code*

Apply inverted dropout with p = 0.5 to a vector of ones 100,000 times. Check that the average output is still 1. Then show what goes wrong without the division.

<details><summary><b>Hint</b></summary>

mask = rng.random(n) > p; out = h * mask / (1 − p).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(7)
h = np.ones(100_000); p = 0.5
mask = rng.random(h.size) > p
with_scale = ...  # TODO: inverted dropout: rescale the survivors
without = h * mask
print(f"mean with 1/(1-p): {with_scale.mean():.4f}   mean without: {without.mean():.4f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(7)
h = np.ones(100_000); p = 0.5
mask = rng.random(h.size) > p
with_scale = h * mask / (1 - p)  # inverted dropout: rescale the survivors
without = h * mask
print(f"mean with 1/(1-p): {with_scale.mean():.4f}   mean without: {without.mean():.4f}")

<details><summary><b>Answer and explanation</b></summary>

With the division the average is ≈ 1, so a neuron downstream sees the same expected input in training and at test time (when dropout is off). Without it the average is ≈ 0.5, and the network would see inputs twice as large at test time as it was trained on (Murphy §13.5.4).

Expected output:
```
mean with 1/(1-p): 1.0031   mean without: 0.5016
```

</details>

## 13.4 · Backprop for one hidden layer, in matrices  
*Section §13.3 · derive*

For h = ReLU(XW₁ + b₁), ŷ = hW₂ + b₂, L = ‖ŷ − Y‖²/(2N), derive ∂L/∂W₂, ∂L/∂h, ∂L/∂W₁. Check that the shapes match.

<details><summary><b>Hint</b></summary>

Start from ∂L/∂ŷ = (ŷ − Y)/N and use station 4's rules: multiply passes blame by the other input; ReLU passes it only where the unit was on.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

δ₂ = ∂L/∂ŷ = (ŷ − Y)/N (N×K). ∂L/∂W₂ = hᵀδ₂ (H×K, matches W₂). ∂L/∂b₂ = Σ rows of δ₂. ∂L/∂h = δ₂W₂ᵀ (N×H). Through ReLU: δ₁ = (δ₂W₂ᵀ) ⊙ 𝟙(XW₁ + b₁ > 0). ∂L/∂W₁ = Xᵀδ₁ (D×H, matches W₁). Every weight gradient is (input to that layer)ᵀ × (blame at its output), which is the matrix version of 'times the other input' (Murphy §13.3.2).

</details>